In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score, roc_auc_score, average_precision_score
)
from scipy.stats import pearsonr
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

In [3]:
# Constants
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
ANNOT_DIR = '/kaggle/input/dataset1/annotations/annotations'
IMG_DIR = '/kaggle/input/dataset1/images/images'

In [ ]:
# Load Annotations
def load_all_annotations(annotation_dir, img_dir, start_idx=0, end_idx=5495):
    data = {
        'file_idx': [], 'expression': [], 'arousal': [], 'valence': [], 'landmarks': [], 'image_path': []
    }
    
    for idx in range(start_idx, end_idx + 1):
        try:
            aro_file = os.path.join(annotation_dir, f"{idx}_aro.npy")
            val_file = os.path.join(annotation_dir, f"{idx}_val.npy")
            exp_file = os.path.join(annotation_dir, f"{idx}_exp.npy")
            lnd_file = os.path.join(annotation_dir, f"{idx}_lnd.npy")
            img_file = os.path.join(img_dir, f"{idx}.jpg")
            
            if not os.path.exists(img_file):
                continue

            aro_value, val_value, exp_value, lnd_value = None, None, None, None

            if os.path.exists(aro_file):
                aro_value = float(np.load(aro_file, allow_pickle=True))
            if os.path.exists(val_file):
                val_value = float(np.load(val_file, allow_pickle=True))
            if os.path.exists(exp_file):
                exp_value = int(np.load(exp_file, allow_pickle=True))
            if os.path.exists(lnd_file):
                lnd_value = np.load(lnd_file, allow_pickle=True)

            if exp_value is not None:
                data['file_idx'].append(idx)
                data['arousal'].append(aro_value)
                data['valence'].append(val_value)
                data['expression'].append(exp_value)
                data['landmarks'].append(lnd_value)
                data['image_path'].append(img_file)
        
        except Exception as e:
            print(f"Error loading {idx}: {e}")

    df = pd.DataFrame(data)
    print("=== Annotation Loading Summary ===")
    print(f"Total Samples: {len(df)}")
    print(f"Expressions: {df['expression'].nunique()} unique")
    print(f"Arousal Range: {df['arousal'].min():.2f} to {df['arousal'].max():.2f}")
    print(f"Valence Range: {df['valence'].min():.2f} to {df['valence'].max():.2f}")
    return df

In [ ]:
# Data Loading and Splitting
def load_data():
    train_df = load_all_annotations(ANNOT_DIR, IMG_DIR, start_idx=0, end_idx=3999)
    test_df = load_all_annotations(ANNOT_DIR, IMG_DIR, start_idx=4000, end_idx=5495)

    # Filter invalid samples
    train_df = train_df[(train_df['valence'] != -2) & (train_df['arousal'] != -2)]
    test_df = test_df[(test_df['valence'] != -2) & (test_df['arousal'] != -2)]

    # Split train into train/val
    train_df, val_df = train_test_split(train_df, test_size=0.2, stratify=train_df['expression'], random_state=42)

    return train_df, val_df, test_df

In [ ]:
# Custom Dataset
class AffectDataset(Dataset):
    def __init__(self, annot_df, img_dir, transform=None):
        self.annot_df = annot_df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.annot_df)

    def __getitem__(self, idx):
        row = self.annot_df.iloc[idx]
        img_path = row['image_path']
        image = Image.open(img_path).convert('RGB')
        label = int(row['expression'])
        valence = float(row['valence'])
        arousal = float(row['arousal'])
        va = torch.tensor([valence, arousal], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, label, va

In [ ]:
# Transforms
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Multi-Task Model
class MultiTaskCNN(nn.Module):
    def __init__(self, base_model, num_classes=8):
        super(MultiTaskCNN, self).__init__()
        self.base = base_model
        self.class_head = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(self._get_feature_size(), num_classes)
        )
        self.reg_head = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(self._get_feature_size(), 2),
            nn.Tanh()
        )

    def _get_feature_size(self):
        with torch.no_grad():
            dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
            features = self.base(dummy)
            return features.view(1, -1).size(1)

    def forward(self, x):
        features = self.base(x)
        features = features.view(features.size(0), -1)
        class_out = self.class_head(features)
        reg_out = self.reg_head(features)
        return class_out, reg_out

In [ ]:
# Get Base Models
def get_resnet50():
    model = models.resnet50(pretrained=True)
    model.fc = nn.Identity()
    return MultiTaskCNN(model)

def get_efficientnet_b0():
    model = models.efficientnet_b0(pretrained=True)
    model.classifier = nn.Identity()
    return MultiTaskCNN(model)

In [ ]:
# Compute Class Weights
def get_class_weights(df):
    class_counts = df['expression'].value_counts().sort_index()
    weights = len(df) / (8 * class_counts)
    return torch.tensor(weights.values, dtype=torch.float32).to(DEVICE)

# Krippendorff's Alpha Implementation
def krippendorff_alpha(data, level_of_measurement='nominal'):
    n = data.shape[1]
    observed = np.zeros((8, 8))
    for i in range(n):
        rater1, rater2 = data[0, i], data[1, i]
        observed[rater1, rater2] += 1
    observed += observed.T - np.diag(np.diag(observed))
    total = np.sum(observed)
    pa = np.sum(np.diag(observed)) / total
    pe = np.sum(np.sum(observed, axis=1) * np.sum(observed, axis=0)) / (total ** 2)
    return (pa - pe) / (1 - pe) if pe != 1 else 1.0

In [ ]:
# Training Function
def train_model(model, train_loader, val_loader, class_weights, model_name):
    criterion_class = nn.CrossEntropyLoss(weight=class_weights)
    criterion_reg = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, verbose=True)

    best_val_loss = float('inf')
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for images, labels, vas in train_loader:
            images, labels, vas = images.to(DEVICE), labels.to(DEVICE), vas.to(DEVICE)
            optimizer.zero_grad()
            class_out, reg_out = model(images)
            loss_class = criterion_class(class_out, labels)
            loss_reg = criterion_reg(reg_out, vas)
            loss = loss_class + loss_reg
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        val_loss, _, _, _, _ = evaluate_model(model, val_loader, criterion_class, criterion_reg)
        scheduler.step(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f'best_model_{model_name}.pth')

        print(f'{model_name} Epoch {epoch+1}/{EPOCHS}, Train Loss: {train_loss/len(train_loader):.4f}, Val Loss: {val_loss:.4f}')

    model.load_state_dict(torch.load(f'best_model_{model_name}.pth'))
    return model

In [ ]:
# Evaluation Function
def evaluate_model(model, loader, criterion_class, criterion_reg):
    model.eval()
    total_loss = 0
    all_class_preds, all_labels = [], []
    all_va_preds, all_vas = [], []
    with torch.no_grad():
        for images, labels, vas in loader:
            images, labels, vas = images.to(DEVICE), labels.to(DEVICE), vas.to(DEVICE)
            class_out, reg_out = model(images)
            loss_class = criterion_class(class_out, labels)
            loss_reg = criterion_reg(reg_out, vas)
            loss = loss_class + loss_reg
            total_loss += loss.item()

            all_class_preds.extend(torch.argmax(class_out, dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_va_preds.extend(reg_out.cpu().numpy())
            all_vas.extend(vas.cpu().numpy())

    return total_loss / len(loader), np.array(all_class_preds), np.array(all_labels), np.array(all_va_preds), np.array(all_vas)


In [4]:


# Compute Metrics
def compute_class_metrics(y_true, y_pred, y_scores=None):
    metrics = {}
    metrics['Accuracy'] = accuracy_score(y_true, y_pred)
    metrics['F1-Score'] = f1_score(y_true, y_pred, average='macro')
    metrics['Cohen Kappa'] = cohen_kappa_score(y_true, y_pred)
    metrics['Krippendorff Alpha'] = krippendorff_alpha(np.stack([y_true, y_pred]))
    
    if y_scores is not None:
        # AUC-ROC (multiclass supported)
        metrics['AUC-ROC'] = roc_auc_score(y_true, y_scores, multi_class='ovr')
        
        # AUC-PR: Compute for each class (one-vs-rest) and average
        n_classes = y_scores.shape[1]
        auc_pr_scores = []
        for i in range(n_classes):
            y_true_binary = (y_true == i).astype(int)
            y_score_class = y_scores[:, i]
            try:
                auc_pr = average_precision_score(y_true_binary, y_score_class)
                auc_pr_scores.append(auc_pr)
            except ValueError:
                continue  # Skip classes with no positive samples
        metrics['AUC-PR'] = np.mean(auc_pr_scores) if auc_pr_scores else 0.0
    
    return metrics

def compute_reg_metrics(va_true, va_pred):
    metrics = {'Valence': {}, 'Arousal': {}}
    for i, key in enumerate(metrics.keys()):
        true = va_true[:, i]
        pred = va_pred[:, i]
        metrics[key]['RMSE'] = np.sqrt(mean_squared_error(true, pred))
        metrics[key]['Correlation'] = pearsonr(true, pred)[0]
        metrics[key]['Sign Agreement'] = np.mean(np.sign(true) == np.sign(pred))
        rho = metrics[key]['Correlation']
        mu_t, mu_p = np.mean(true), np.mean(pred)
        sigma_t, sigma_p = np.std(true), np.std(pred)
        metrics[key]['CCC'] = (2 * rho * sigma_t * sigma_p) / (sigma_t**2 + sigma_p**2 + (mu_t - mu_p)**2)
    return metrics


In [5]:
train_df, val_df, test_df = load_data()

=== Annotation Loading Summary ===
Total Samples: 2919
Expressions: 8 unique
Arousal Range: -0.67 to 0.98
Valence Range: -0.99 to 0.98
=== Annotation Loading Summary ===
Total Samples: 1080
Expressions: 8 unique
Arousal Range: -0.67 to 0.98
Valence Range: -0.96 to 0.96


In [6]:
class_weights = get_class_weights(train_df)

In [7]:
train_ds = AffectDataset(train_df, IMG_DIR, train_transform)
val_ds = AffectDataset(val_df, IMG_DIR, val_test_transform)
test_ds = AffectDataset(test_df, IMG_DIR, val_test_transform)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, BATCH_SIZE)
test_loader = DataLoader(test_ds, BATCH_SIZE)

In [8]:
models_dict = {
    'ResNet50': get_resnet50().to(DEVICE),
    'EfficientNetB0': get_efficientnet_b0().to(DEVICE)
}

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 201MB/s] 
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed 

In [9]:


results = {}
for name, model in models_dict.items():
    print(f'Training {name}...')
    criterion_class = nn.CrossEntropyLoss(weight=class_weights)
    criterion_reg = nn.MSELoss()
    trained_model = train_model(model, train_loader, val_loader, class_weights, name)
    
    print(f'Evaluating {name} on Test Set...')
    _, class_preds, labels, va_preds, vas = evaluate_model(trained_model, test_loader, criterion_class, criterion_reg)
    
    all_scores = []
    with torch.no_grad():
        for images, _, _ in test_loader:
            images = images.to(DEVICE)
            class_out, _ = trained_model(images)
            all_scores.extend(torch.softmax(class_out, dim=1).cpu().numpy())
    all_scores = np.array(all_scores)
    
    class_metrics = compute_class_metrics(labels, class_preds, all_scores)
    reg_metrics = compute_reg_metrics(vas, va_preds)
    results[name] = {'Classification': class_metrics, 'Regression': reg_metrics}




Training ResNet50...


/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


ResNet50 Epoch 1/20, Train Loss: 2.1598, Val Loss: 1.8316
ResNet50 Epoch 2/20, Train Loss: 1.7002, Val Loss: 1.7196
ResNet50 Epoch 3/20, Train Loss: 1.4044, Val Loss: 1.8142
ResNet50 Epoch 4/20, Train Loss: 1.1675, Val Loss: 1.8730
ResNet50 Epoch 5/20, Train Loss: 0.9531, Val Loss: 1.8937
ResNet50 Epoch 6/20, Train Loss: 0.7735, Val Loss: 2.3959
ResNet50 Epoch 7/20, Train Loss: 0.5427, Val Loss: 2.0015
ResNet50 Epoch 8/20, Train Loss: 0.4174, Val Loss: 1.9934
ResNet50 Epoch 9/20, Train Loss: 0.3674, Val Loss: 1.9896
ResNet50 Epoch 10/20, Train Loss: 0.3262, Val Loss: 2.0326
ResNet50 Epoch 11/20, Train Loss: 0.2953, Val Loss: 2.0070
ResNet50 Epoch 12/20, Train Loss: 0.2920, Val Loss: 2.0138
ResNet50 Epoch 13/20, Train Loss: 0.2880, Val Loss: 2.0209
ResNet50 Epoch 14/20, Train Loss: 0.2817, Val Loss: 2.0263
ResNet50 Epoch 15/20, Train Loss: 0.2959, Val Loss: 2.0401
ResNet50 Epoch 16/20, Train Loss: 0.2814, Val Loss: 2.0248
ResNet50 Epoch 17/20, Train Loss: 0.2795, Val Loss: 2.0328
ResNet

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


EfficientNetB0 Epoch 1/20, Train Loss: 2.3049, Val Loss: 2.2090
EfficientNetB0 Epoch 2/20, Train Loss: 2.1248, Val Loss: 2.0918
EfficientNetB0 Epoch 3/20, Train Loss: 1.9326, Val Loss: 1.9387
EfficientNetB0 Epoch 4/20, Train Loss: 1.7674, Val Loss: 1.8578
EfficientNetB0 Epoch 5/20, Train Loss: 1.5900, Val Loss: 1.8189
EfficientNetB0 Epoch 6/20, Train Loss: 1.4574, Val Loss: 1.7816
EfficientNetB0 Epoch 7/20, Train Loss: 1.3037, Val Loss: 1.7866
EfficientNetB0 Epoch 8/20, Train Loss: 1.1848, Val Loss: 1.8044
EfficientNetB0 Epoch 9/20, Train Loss: 1.0185, Val Loss: 1.8673
EfficientNetB0 Epoch 10/20, Train Loss: 0.8883, Val Loss: 1.9051
EfficientNetB0 Epoch 11/20, Train Loss: 0.7537, Val Loss: 1.9223
EfficientNetB0 Epoch 12/20, Train Loss: 0.7360, Val Loss: 1.9367
EfficientNetB0 Epoch 13/20, Train Loss: 0.7308, Val Loss: 1.9504
EfficientNetB0 Epoch 14/20, Train Loss: 0.7080, Val Loss: 1.9514
EfficientNetB0 Epoch 15/20, Train Loss: 0.6995, Val Loss: 1.9428
EfficientNetB0 Epoch 16/20, Train 

In [10]:
print('\nClassification Comparison:')
class_df = pd.DataFrame({k: v['Classification'] for k, v in results.items()}).T
print(class_df)

print('\nRegression Comparison:')
for key in ['Valence', 'Arousal']:
    reg_df = pd.DataFrame({k: v['Regression'][key] for k, v in results.items()}).T
    print(f'{key}:\n{reg_df}\n')


Classification Comparison:
                Accuracy  F1-Score  Cohen Kappa  Krippendorff Alpha   AUC-ROC  \
ResNet50        0.392593  0.387018     0.305250            0.130147  0.814755   
EfficientNetB0  0.405556  0.404478     0.320144            0.146265  0.812074   

                  AUC-PR  
ResNet50        0.424866  
EfficientNetB0  0.420065  

Regression Comparison:
Valence:
                    RMSE  Correlation  Sign Agreement       CCC
ResNet50        0.415037     0.474977        0.726852  0.443982
EfficientNetB0  0.426206     0.463513        0.700926  0.441326

Arousal:
                    RMSE  Correlation  Sign Agreement       CCC
ResNet50        0.358427     0.404951        0.755556  0.373185
EfficientNetB0  0.358805     0.396616        0.756481  0.348291



//////////////////

In [6]:


# Focal Loss for Classification
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha  # Class weights
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma * ce_loss).mean()
        return focal_loss

# Load Annotations
def load_all_annotations(annotation_dir, img_dir, start_idx=0, end_idx=5495):
    data = {
        'file_idx': [], 'expression': [], 'arousal': [], 'valence': [], 'landmarks': [], 'image_path': []
    }
    
    for idx in range(start_idx, end_idx + 1):
        try:
            aro_file = os.path.join(annotation_dir, f"{idx}_aro.npy")
            val_file = os.path.join(annotation_dir, f"{idx}_val.npy")
            exp_file = os.path.join(annotation_dir, f"{idx}_exp.npy")
            lnd_file = os.path.join(annotation_dir, f"{idx}_lnd.npy")
            img_file = os.path.join(img_dir, f"{idx}.jpg")
            
            if not os.path.exists(img_file):
                continue

            aro_value, val_value, exp_value, lnd_value = None, None, None, None

            if os.path.exists(aro_file):
                aro_value = float(np.load(aro_file, allow_pickle=True))
            if os.path.exists(val_file):
                val_value = float(np.load(val_file, allow_pickle=True))
            if os.path.exists(exp_file):
                exp_value = int(np.load(exp_file, allow_pickle=True))
            if os.path.exists(lnd_file):
                lnd_value = np.load(lnd_file, allow_pickle=True)

            if exp_value is not None:
                data['file_idx'].append(idx)
                data['arousal'].append(aro_value)
                data['valence'].append(val_value)
                data['expression'].append(exp_value)
                data['landmarks'].append(lnd_value)
                data['image_path'].append(img_file)
        
        except Exception as e:
            print(f"Error loading {idx}: {e}")

    df = pd.DataFrame(data)
    print("=== Annotation Loading Summary ===")
    print(f"Total Samples: {len(df)}")
    print(f"Expressions: {df['expression'].nunique()} unique")
    print(f"Expression Distribution:\n{df['expression'].value_counts()}")
    print(f"Arousal Range: {df['arousal'].min():.2f} to {df['arousal'].max():.2f}")
    print(f"Valence Range: {df['valence'].min():.2f} to {df['valence'].max():.2f}")
    return df

# Data Loading and Splitting
def load_data():
    train_df = load_all_annotations(ANNOT_DIR, IMG_DIR, start_idx=0, end_idx=3999)
    test_df = load_all_annotations(ANNOT_DIR, IMG_DIR, start_idx=4000, end_idx=5495)

    train_df = train_df[(train_df['valence'] != -2) & (train_df['arousal'] != -2)]
    test_df = test_df[(test_df['valence'] != -2) & (test_df['arousal'] != -2)]

    train_df, val_df = train_test_split(train_df, test_size=0.2, stratify=train_df['expression'], random_state=42)

    print("\n=== Dataset Split Summary ===")
    print(f"Training Samples: {len(train_df)}")
    print(f"Validation Samples: {len(val_df)}")
    print(f"Test Samples: {len(test_df)}")
    print(f"Train Expression Distribution:\n{train_df['expression'].value_counts()}")
    print(f"Test Expression Distribution:\n{test_df['expression'].value_counts()}")

    return train_df, val_df, test_df

# Custom Dataset
class AffectDataset(Dataset):
    def __init__(self, annot_df, img_dir, transform=None):
        self.annot_df = annot_df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.annot_df)

    def __getitem__(self, idx):
        row = self.annot_df.iloc[idx]
        img_path = row['image_path']
        image = Image.open(img_path).convert('RGB')
        label = int(row['expression'])
        valence = float(row['valence'])
        arousal = float(row['arousal'])
        va = torch.tensor([valence, arousal], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, label, va

# Transforms
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),  # Reduced rotation
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Multi-Task Model
class MultiTaskCNN(nn.Module):
    def __init__(self, base_model, num_classes=8):
        super(MultiTaskCNN, self).__init__()
        self.base = base_model
        self.class_head = nn.Sequential(
            nn.Dropout(0.3),  # Reduced dropout
            nn.Linear(self._get_feature_size(), num_classes)
        )
        self.reg_head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self._get_feature_size(), 2),
            nn.Tanh()
        )

    def _get_feature_size(self):
        with torch.no_grad():
            dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
            features = self.base(dummy)
            return features.view(1, -1).size(1)

    def forward(self, x):
        features = self.base(x)
        features = features.view(features.size(0), -1)
        class_out = self.class_head(features)
        reg_out = self.reg_head(features)
        return class_out, reg_out

# Get Base Models
def get_resnet50():
    model = models.resnet50(pretrained=True)
    model.fc = nn.Identity()
    return MultiTaskCNN(model)

def get_efficientnet_b0():
    model = models.efficientnet_b0(pretrained=True)
    model.classifier = nn.Identity()
    return MultiTaskCNN(model)

# Compute Class Weights
def get_class_weights(df):
    class_counts = df['expression'].value_counts().sort_index()
    weights = len(df) / (8 * class_counts)
    return torch.tensor(weights.values, dtype=torch.float32).to(DEVICE)

# Krippendorff's Alpha
def krippendorff_alpha(data, level_of_measurement='nominal'):
    n = data.shape[1]
    observed = np.zeros((8, 8))
    for i in range(n):
        rater1, rater2 = data[0, i], data[1, i]
        observed[rater1, rater2] += 1
    observed += observed.T - np.diag(np.diag(observed))
    total = np.sum(observed)
    pa = np.sum(np.diag(observed)) / total
    pe = np.sum(np.sum(observed, axis=1) * np.sum(observed, axis=0)) / (total ** 2)
    return (pa - pe) / (1 - pe) if pe != 1 else 1.0

# Training Function
def train_model(model, train_loader, val_loader, class_weights, model_name):
    criterion_class = FocalLoss(gamma=2.0, alpha=class_weights)
    criterion_reg = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5, verbose=True)

    # Freeze backbone initially
    for param in model.base.parameters():
        param.requires_grad = False

    best_val_loss = float('inf')
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for images, labels, vas in train_loader:
            images, labels, vas = images.to(DEVICE), labels.to(DEVICE), vas.to(DEVICE)
            optimizer.zero_grad()
            class_out, reg_out = model(images)
            loss_class = criterion_class(class_out, labels)
            loss_reg = criterion_reg(reg_out, vas)
            loss = 0.7 * loss_class + 0.3 * loss_reg  # Weight classification higher
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Unfreeze backbone after 10 epochs
        if epoch == 10:
            for param in model.base.parameters():
                param.requires_grad = True
            optimizer = optim.AdamW(model.parameters(), lr=1e-4)  # Lower LR for fine-tuning

        val_loss, _, _, _, _ = evaluate_model(model, val_loader, criterion_class, criterion_reg)
        scheduler.step(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f'best_model_{model_name}.pth')

        print(f'{model_name} Epoch {epoch+1}/{EPOCHS}, Train Loss: {train_loss/len(train_loader):.4f}, Val Loss: {val_loss:.4f}')

    model.load_state_dict(torch.load(f'best_model_{model_name}.pth'))
    return model

# Evaluation Function
def evaluate_model(model, loader, criterion_class, criterion_reg):
    model.eval()
    total_loss = 0
    all_class_preds, all_labels = [], []
    all_va_preds, all_vas = [], []
    with torch.no_grad():
        for images, labels, vas in loader:
            images, labels, vas = images.to(DEVICE), labels.to(DEVICE), vas.to(DEVICE)
            class_out, reg_out = model(images)
            loss_class = criterion_class(class_out, labels)
            loss_reg = criterion_reg(reg_out, vas)
            loss = 0.7 * loss_class + 0.3 * loss_reg
            total_loss += loss.item()

            all_class_preds.extend(torch.argmax(class_out, dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_va_preds.extend(reg_out.cpu().numpy())
            all_vas.extend(vas.cpu().numpy())

    return total_loss / len(loader), np.array(all_class_preds), np.array(all_labels), np.array(all_va_preds), np.array(all_vas)

# Compute Metrics
def compute_class_metrics(y_true, y_pred, y_scores=None):
    metrics = {}
    metrics['Accuracy'] = accuracy_score(y_true, y_pred)
    metrics['F1-Score'] = f1_score(y_true, y_pred, average='macro')
    metrics['Cohen Kappa'] = cohen_kappa_score(y_true, y_pred)
    metrics['Krippendorff Alpha'] = krippendorff_alpha(np.stack([y_true, y_pred]))
    
    if y_scores is not None:
        metrics['AUC-ROC'] = roc_auc_score(y_true, y_scores, multi_class='ovr')
        n_classes = y_scores.shape[1]
        auc_pr_scores = []
        for i in range(n_classes):
            y_true_binary = (y_true == i).astype(int)
            y_score_class = y_scores[:, i]
            try:
                auc_pr = average_precision_score(y_true_binary, y_score_class)
                auc_pr_scores.append(auc_pr)
            except ValueError:
                continue
        metrics['AUC-PR'] = np.mean(auc_pr_scores) if auc_pr_scores else 0.0
    
    return metrics

def compute_reg_metrics(va_true, va_pred):
    metrics = {'Valence': {}, 'Arousal': {}}
    for i, key in enumerate(metrics.keys()):
        true = va_true[:, i]
        pred = va_pred[:, i]
        metrics[key]['RMSE'] = np.sqrt(mean_squared_error(true, pred))
        metrics[key]['Correlation'] = pearsonr(true, pred)[0]
        metrics[key]['Sign Agreement'] = np.mean(np.sign(true) == np.sign(pred))
        rho = metrics[key]['Correlation']
        mu_t, mu_p = np.mean(true), np.mean(pred)
        sigma_t, sigma_p = np.std(true), np.std(pred)
        metrics[key]['CCC'] = (2 * rho * sigma_t * sigma_p) / (sigma_t**2 + sigma_p**2 + (mu_t - mu_p)**2)
    return metrics



In [7]:

# Main Function
def main():
    train_df, val_df, test_df = load_data()
    class_weights = get_class_weights(train_df)

    train_ds = AffectDataset(train_df, IMG_DIR, train_transform)
    val_ds = AffectDataset(val_df, IMG_DIR, val_test_transform)
    test_ds = AffectDataset(test_df, IMG_DIR, val_test_transform)

    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, BATCH_SIZE)
    test_loader = DataLoader(test_ds, BATCH_SIZE)

    models_dict = {
        'ResNet50': get_resnet50().to(DEVICE),
        'EfficientNetB0': get_efficientnet_b0().to(DEVICE)
    }

    results = {}
    for name, model in models_dict.items():
        print(f'Training {name}...')
        criterion_class = FocalLoss(gamma=2.0, alpha=class_weights)
        criterion_reg = nn.MSELoss()
        trained_model = train_model(model, train_loader, val_loader, class_weights, name)
        
        print(f'Evaluating {name} on Test Set...')
        _, class_preds, labels, va_preds, vas = evaluate_model(trained_model, test_loader, criterion_class, criterion_reg)
        
        all_scores = []
        with torch.no_grad():
            for images, _, _ in test_loader:
                images = images.to(DEVICE)
                class_out, _ = trained_model(images)
                all_scores.extend(torch.softmax(class_out, dim=1).cpu().numpy())
        all_scores = np.array(all_scores)
        
        class_metrics = compute_class_metrics(labels, class_preds, all_scores)
        reg_metrics = compute_reg_metrics(vas, va_preds)
        results[name] = {'Classification': class_metrics, 'Regression': reg_metrics}

    print('\nClassification Comparison:')
    class_df = pd.DataFrame({k: v['Classification'] for k, v in results.items()}).T
    print(class_df)

    print('\nRegression Comparison:')
    for key in ['Valence', 'Arousal']:
        reg_df = pd.DataFrame({k: v['Regression'][key] for k, v in results.items()}).T
        print(f'{key}:\n{reg_df}\n')

In [8]:
main()

=== Annotation Loading Summary ===
Total Samples: 2919
Expressions: 8 unique
Expression Distribution:
expression
1    378
6    375
3    369
2    364
4    361
5    361
7    356
0    355
Name: count, dtype: int64
Arousal Range: -0.67 to 0.98
Valence Range: -0.99 to 0.98
=== Annotation Loading Summary ===
Total Samples: 1080
Expressions: 8 unique
Expression Distribution:
expression
0    145
7    143
4    139
5    139
2    136
3    131
6    125
1    122
Name: count, dtype: int64
Arousal Range: -0.67 to 0.98
Valence Range: -0.96 to 0.96

=== Dataset Split Summary ===
Training Samples: 2335
Validation Samples: 584
Test Samples: 1080
Train Expression Distribution:
expression
1    302
6    300
3    295
2    291
5    289
4    289
7    285
0    284
Name: count, dtype: int64
Test Expression Distribution:
expression
0    145
7    143
4    139
5    139
2    136
3    131
6    125
1    122
Name: count, dtype: int64


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 173MB/s] 
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed 

Training ResNet50...


/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


ResNet50 Epoch 1/20, Train Loss: 1.2153, Val Loss: 1.1739
ResNet50 Epoch 2/20, Train Loss: 1.1748, Val Loss: 1.1449
ResNet50 Epoch 3/20, Train Loss: 1.1521, Val Loss: 1.1304
ResNet50 Epoch 4/20, Train Loss: 1.1223, Val Loss: 1.1208
ResNet50 Epoch 5/20, Train Loss: 1.0959, Val Loss: 1.0971
ResNet50 Epoch 6/20, Train Loss: 1.0953, Val Loss: 1.0969
ResNet50 Epoch 7/20, Train Loss: 1.0839, Val Loss: 1.0775
ResNet50 Epoch 8/20, Train Loss: 1.0564, Val Loss: 1.0663
ResNet50 Epoch 9/20, Train Loss: 1.0511, Val Loss: 1.0641
ResNet50 Epoch 10/20, Train Loss: 1.0414, Val Loss: 1.0542
ResNet50 Epoch 11/20, Train Loss: 1.0295, Val Loss: 1.0506
ResNet50 Epoch 12/20, Train Loss: 0.9112, Val Loss: 0.8269
ResNet50 Epoch 13/20, Train Loss: 0.6677, Val Loss: 0.8636
ResNet50 Epoch 14/20, Train Loss: 0.5043, Val Loss: 0.9240
ResNet50 Epoch 15/20, Train Loss: 0.3861, Val Loss: 0.9491
ResNet50 Epoch 16/20, Train Loss: 0.3006, Val Loss: 1.0180
ResNet50 Epoch 17/20, Train Loss: 0.2253, Val Loss: 0.9727
ResNet

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


EfficientNetB0 Epoch 1/20, Train Loss: 1.1866, Val Loss: 1.1767
EfficientNetB0 Epoch 2/20, Train Loss: 1.1615, Val Loss: 1.1597
EfficientNetB0 Epoch 3/20, Train Loss: 1.1386, Val Loss: 1.1452
EfficientNetB0 Epoch 4/20, Train Loss: 1.1242, Val Loss: 1.1352
EfficientNetB0 Epoch 5/20, Train Loss: 1.1089, Val Loss: 1.1225
EfficientNetB0 Epoch 6/20, Train Loss: 1.0891, Val Loss: 1.1136
EfficientNetB0 Epoch 7/20, Train Loss: 1.0779, Val Loss: 1.1085
EfficientNetB0 Epoch 8/20, Train Loss: 1.0610, Val Loss: 1.0984
EfficientNetB0 Epoch 9/20, Train Loss: 1.0519, Val Loss: 1.0923
EfficientNetB0 Epoch 10/20, Train Loss: 1.0437, Val Loss: 1.0845
EfficientNetB0 Epoch 11/20, Train Loss: 1.0324, Val Loss: 1.0819
EfficientNetB0 Epoch 12/20, Train Loss: 0.9685, Val Loss: 0.9659
EfficientNetB0 Epoch 13/20, Train Loss: 0.8232, Val Loss: 0.9054
EfficientNetB0 Epoch 14/20, Train Loss: 0.7261, Val Loss: 0.8765
EfficientNetB0 Epoch 15/20, Train Loss: 0.6278, Val Loss: 0.8780
EfficientNetB0 Epoch 16/20, Train 